In [1]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, accuracy_score
import numpy as np
import xgboost as xgb
import json
import pandas as pd
from pandas import json_normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from textblob import TextBlob
import re
from sklearn.ensemble import RandomForestClassifier
#from gensim.models import Word2Vec
#from gensim.models import Word2Vec

In [3]:
# ===============================
# JSONL LOADING
# ===============================


# Load the training data from a JSON Lines file (one JSON object per line)
import json
from pandas import json_normalize

def load_jsonl_skip_bad(path):
    data_list = []
    with open(path, "r") as f:
        for line in f:
            try:
                data_list.append(json.loads(line))
            except json.JSONDecodeError:
                # Skip bad JSON line silently
                continue
    return json_normalize(data_list)

# Usage:
#train_data = load_jsonl_skip_bad('train.jsonl')
train_data = pd.read_json('train.jsonl',lines="True")
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
#kaggle_data = load_jsonl_skip_bad('kaggle_test.jsonl')
kaggle_data = pd.read_json('kaggle_test.jsonl',lines="True")
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


# Separate features from the target variable for the training set
X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

print("✓ Chargement OK")

✓ Chargement OK


In [4]:
import pandas as pd
import numpy as np
from datetime import datetime

def create_advanced_features(df_input):
    df = df_input.copy()

    # Définition des séries de fallback (robustesse contre les colonnes manquantes)
    default_int_series = pd.Series(0, index=df.index)
    default_bool_series = pd.Series(False, index=df.index)

    # --- Initialisation des colonnes numériques ---
    df['user.followers_count'] = df.get('user.followers_count', default_int_series).fillna(0)
    df['user.friends_count'] = df.get('user.friends_count', default_int_series).fillna(0)
    df['user.listed_count'] = df.get('user.listed_count', default_int_series).fillna(0)
    df['user.favourites_count'] = df.get('user.favourites_count', default_int_series).fillna(0)
    df['user.statuses_count'] = df.get('user.statuses_count', default_int_series).fillna(0)
    df['retweet_count'] = df.get('retweet_count', default_int_series).fillna(0)
    df['favorite_count'] = df.get('favorite_count', default_int_series).fillna(0)
    df['quote_count'] = df.get('quote_count', default_int_series).fillna(0) # Nouveau : Quote Count
    df['reply_count'] = df.get('reply_count', default_int_series).fillna(0) # Nouveau : Reply Count

    # --- A. GESTION DES DATES (Ancienneté du compte) ---
    df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
    ref_date = pd.to_datetime('now', utc=True)
    df['account_age_days'] = (ref_date - df['user_created_at_dt']).dt.days
    df['account_age_days'] = df['account_age_days'].fillna(0)

    # Extraction des indicateurs temporels avancés du tweet
    df['created_at_dt'] = pd.to_datetime(df.get('created_at'), errors='coerce')
    df['tweet_hour'] = df['created_at_dt'].dt.hour.fillna(-1)
    df['tweet_is_weekend'] = df['created_at_dt'].dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    # --- B. QUALITÉ DU PROFIL & STATUT (Booléens) ---
    df['is_default_profile'] = df.get('user.default_profile', default_bool_series).fillna(False).astype(int)
    df['is_default_image'] = df.get('user.default_profile_image', default_bool_series).fillna(False).astype(int)
    df['is_verified'] = df.get('user.verified', default_bool_series).fillna(False).astype(int)

    # Nouveau : Est-ce un compte protégé/privé ? (Signe d'un follower ou d'un utilisateur personnel)
    df['is_protected'] = df.get('user.protected', default_bool_series).fillna(False).astype(int)

    # Nouveau : Le profil a-t-il une URL renseignée ?
    df['has_url'] = df.get('user.url', pd.Series(False, index=df.index)).notna().astype(int)

    # --- C. CONTENU DU TWEET (Entities Counting) ---
    def count_entities(x):
        if isinstance(x, list) or (isinstance(x, pd.Series) and x.dtype == object): return len(x)
        return 0

    df['num_urls'] = df.get('entities.urls', default_int_series).apply(count_entities)
    df['num_hashtags'] = df.get('entities.hashtags', default_int_series).apply(count_entities)
    df['num_mentions'] = df.get('entities.user_mentions', default_int_series).apply(count_entities)
    df['has_media'] = df.get('extended_entities.media', default_bool_series).notna().astype(int)

    # --- D. RATIOS PUISSANTS & COMPORTEMENTAUX ---
    followers = df['user.followers_count']
    friends = df['user.friends_count']
    listed = df['user.listed_count']
    statuses = df['user.statuses_count']

    # 1. Ratio Followers / Friends (Ratio de notoriété)
    df['ratio_followers_friends'] = followers / (friends + 1)
    df['ratio_listed_followers'] = listed / (followers + 1)

    # 2. Taux de Réciprocité d'Amitié (Nouveau - Indique un équilibre/déséquilibre d'influence)
    df['reciprocity_score'] = (friends - followers) / (friends + followers + 1)

    # 3. Activité (Tweets par jour d'existence)
    df['tweets_per_day'] = statuses / (df['account_age_days'] + 1)

    # 4. Ratio Mention/Tweet (Nouveau - Taux d'interaction vs. diffusion)
    # Plus ce ratio est élevé, plus l'utilisateur interagit personnellement (follower).
    df['ratio_mention_status'] = df['num_mentions'] / (statuses + 1)

    # 5. Engagement (Taux d'engagement par Tweet)
    total_engagement = df['retweet_count'] + df['favorite_count'] + df['quote_count'] + df['reply_count']
    df['total_tweet_engagement'] = total_engagement / (followers + 1)

    # --- E. LONGUEUR DES TEXTES ---
    df['final_text'] = df.get('extended_tweet.full_text', df.get('text', pd.Series(''))).fillna('')
    df['final_text'] = df['final_text'].where(df['final_text'] != '', df.get('text', '')).fillna('')

    df['text_length'] = df['final_text'].astype(str).apply(len)
    df['bio_length'] = df.get('user.description', '').astype(str).apply(len)

    # --- F. SÉLECTION FINALE ---
    features_to_keep = [
        # Métriques User Brutes
        'user.followers_count', 'user.friends_count', 'user.listed_count',
        'user.favourites_count', 'user.statuses_count',
        # Métriques Tweet Brutes
        'retweet_count', 'favorite_count', 'quote_count', 'reply_count',
        # Ratios & Comportementaux (NOUVEAU)
        'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days',
        'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement',
        # Booléens & Qualité (MIS À JOUR)
        'is_verified', 'is_default_profile', 'is_default_image', 'is_geo_enabled',
        'is_protected', 'has_url',
        # Temporels (NOUVEAU)
        'tweet_hour', 'tweet_is_weekend',
        # Longueur du Contenu
        'text_length', 'bio_length',
        # Compte des Entités
        'num_urls', 'num_hashtags', 'num_mentions', 'has_media',
    ]

    final_cols = [c for c in features_to_keep if c in df.columns]

    return df[final_cols].fillna(0)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from textblob import TextBlob
import re
from nltk.corpus import stopwords

def create_nlp_features(df_train, df_test, y_train):
    """
    Crée des features NLP (TF-IDF des Bios et Sentiment des Tweets).

    Args:
        df_train (pd.DataFrame): DataFrame d'entraînement complet (X_train).
        df_test (pd.DataFrame): DataFrame de test (X_kaggle).
        y_train (pd.Series): Cible d'entraînement (y_train).

    Returns:
        tuple: (df_train_nlp, df_test_nlp) avec les nouvelles colonnes.
    """

    # ----------------------------------------
    # Préparation du texte
    # ----------------------------------------

    french_stopwords = stopwords.words("french")

    # Remplacer les NaN ou valeurs manquantes par une chaîne vide
    train_bio = df_train.get('user.description', pd.Series([''] * len(df_train))).fillna('').astype(str)
    test_bio = df_test.get('user.description', pd.Series([''] * len(df_test))).fillna('').astype(str)

    # Récupération du 'final_text' du tweet (le plus complet)
    # Note: On doit reproduire la logique de 'final_text' de la fonction d'ingénierie
    def get_final_text(df):
        text = df.get('text', pd.Series([''] * len(df))).fillna('')
        full_text = df.get('extended_tweet.full_text', text).fillna(text)
        return full_text.astype(str)

    train_text = get_final_text(df_train)
    test_text = get_final_text(df_test)

    # ----------------------------------------
    # A. TF-IDF sur les tweets (Meta-Feature)
    # ----------------------------------------
    print("TF-IDF vectorization du corps du tweet")

    # Nettoyage très basique du texte pour le TF-IDF
    def clean_text(text):
        #text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Suppression des URLs
        #text = re.sub(r'[^\w\s]', '', text) # Suppression de la ponctuation
        return text

    train_text_clean = train_text.apply(clean_text)
    test_text_clean = test_text.apply(clean_text)

    # 1. Création du Vecteur TF-IDF (Appris uniquement sur le training set)
    tfidf = TfidfVectorizer(max_features=1000, stop_words=french_stopwords, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf_tweet = tfidf.fit_transform(train_text_clean)
    X_test_tfidf_tweet = tfidf.transform(test_text_clean)

    # 2. Entraînement du Méta-Modèle (Régression Logistique)
    log_reg = LogisticRegression(solver='sag', random_state=42)
    log_reg.fit(X_train_tfidf_tweet, y_train.astype(int))

    # 3. Extraction de la probabilité prédite (Meta-Feature)
    # Nous utilisons la probabilité pour la classe 1 (Influencer)
    train_tweet_proba = log_reg.predict_proba(X_train_tfidf_tweet)[:, 1]
    test_tweet_proba = log_reg.predict_proba(X_test_tfidf_tweet)[:, 1]

    # ----------------------------------------
    # B. TF-IDF sur les Bios (Meta-Feature)
    # ----------------------------------------
    print("  -> Calcul du TF-IDF sur les Bios et entraînement du Méta-Modèle...")


    train_bio_clean = train_bio.apply(clean_text)
    test_bio_clean = test_bio.apply(clean_text)

    # 1. Création du Vecteur TF-IDF (Appris uniquement sur le training set)
    tfidf = TfidfVectorizer(max_features=1000, stop_words=french_stopwords, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf = tfidf.fit_transform(train_bio_clean)
    X_test_tfidf = tfidf.transform(test_bio_clean)

    # 2. Entraînement du Méta-Modèle (Régression Logistique)
    log_reg = LogisticRegression(solver='liblinear', random_state=42)
    log_reg.fit(X_train_tfidf, y_train.astype(int))

    # 3. Extraction de la probabilité prédite (Meta-Feature)
    # Nous utilisons la probabilité pour la classe 1 (Influencer)
    train_bio_proba = log_reg.predict_proba(X_train_tfidf)[:, 1]
    test_bio_proba = log_reg.predict_proba(X_test_tfidf)[:, 1]

    # ----------------------------------------
    # B. Analyse du Sentiment (Polarity et Subjectivity)
    # ----------------------------------------
    print("  -> Extraction du Sentiment (Polarity/Subjectivity) des Tweets...")

    # La fonction TextBlob est utilisée pour obtenir les scores de sentiment
    # C'est une opération lente, il faut être patient
    def get_sentiment(text):
        try:
            analysis = TextBlob(text)
            return pd.Series({'polarity': analysis.sentiment.polarity, 'subjectivity': analysis.sentiment.subjectivity})
        except:
            return pd.Series({'polarity': 0.0, 'subjectivity': 0.0})

    train_sentiment = train_text.apply(get_sentiment)
    test_sentiment = test_text.apply(get_sentiment)

    # ----------------------------------------
    # 4. Fusion des Features NLP
    # ----------------------------------------

    # Création des DataFrames de features NLP
    df_train_nlp = pd.DataFrame({
        'meta_bio_proba': train_bio_proba,
        'tweet_polarity': train_sentiment['polarity'],
        'tweet_subjectivity': train_sentiment['subjectivity'],
        'meta_tweet_proba': train_tweet_proba
    })

    df_test_nlp = pd.DataFrame({
        'meta_bio_proba': test_bio_proba,
        'tweet_polarity': test_sentiment['polarity'],
        'tweet_subjectivity': test_sentiment['subjectivity'],
        'meta_tweet_proba': test_tweet_proba
    })

    return df_train_nlp, df_test_nlp

In [16]:
# =======================================================
# On concatène ca dans X_train_advanced et X_kaggle_advanced
# =======================================================
print("🛠️ Construction des features avancées (Métadonnées)...")

import nltk
nltk.download('stopwords')

# Application de la fonction robuste (Métadonnées)
X_train_advanced = create_advanced_features(X_train)
X_kaggle_advanced = create_advanced_features(X_kaggle)

# Préparation de la cible
y_train_clean = y_train.astype(int)

# --- NOUVELLE ÉTAPE : CRÉATION DES FEATURES NLP ---
X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)

# --- FUSION DES FEATURES ---
X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

print(f"\nFeatures combinées ({len(X_train_advanced.columns)}):")
print(list(X_train_advanced.columns))

🛠️ Construction des features avancées (Métadonnées)...


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/tmp/ipython-input-2706182517.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
/tmp/ipython-input-2706182517.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')


TF-IDF vectorization du corps du tweet


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


  -> Calcul du TF-IDF sur les Bios et entraînement du Méta-Modèle...


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


  -> Extraction du Sentiment (Polarity/Subjectivity) des Tweets...

Features combinées (33):
['user.followers_count', 'user.friends_count', 'user.listed_count', 'user.favourites_count', 'user.statuses_count', 'retweet_count', 'favorite_count', 'quote_count', 'reply_count', 'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days', 'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement', 'is_verified', 'is_default_profile', 'is_default_image', 'is_protected', 'has_url', 'tweet_hour', 'tweet_is_weekend', 'text_length', 'bio_length', 'num_urls', 'num_hashtags', 'num_mentions', 'has_media', 'meta_bio_proba', 'tweet_polarity', 'tweet_subjectivity', 'meta_tweet_proba']


In [ ]:
#Heavy unfinetuned version
# ==========================================
# MiniLM + XGBoost TRAINING PIPELINE (FIXED)
# ==========================================

from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV

# ------------------------------------------
# Load MiniLM on GPU
# ------------------------------------------
model_name = "nreimers/MiniLM-L6-H384-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
model.to("cuda")

#TO DO : fetch my own model after unzipping the lora_minilm_out.zip file

# ------------------------------------------
# Pooling function
# ------------------------------------------
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

# ------------------------------------------
# FIXED → Always cast text to str
# ------------------------------------------
def safe_cast_to_str(series):
    return series.fillna("").astype(str).apply(lambda x: str(x))

# ------------------------------------------
# Extract tweet text robustly
# ------------------------------------------
def get_final_text(df):
    if 'extended_tweet.full_text' in df.columns:
        return safe_cast_to_str(df['extended_tweet.full_text'])
    elif 'text' in df.columns:
        return safe_cast_to_str(df['text'])
    else:
        return pd.Series([""] * len(df))

# ------------------------------------------
# MiniLM Embedding Function (Batch GPU)
# ------------------------------------------
def get_miniLM_embeddings(texts, batch_size=64):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = [str(t) for t in texts[i:i+batch_size]]   # FIXED here

        encoded_input = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )

        encoded_input = {k: v.to("cuda") for k, v in encoded_input.items()}

        with torch.no_grad():
            model_output = model(**encoded_input)

        embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings).numpy()

# ------------------------------------------
# Extract clean text
# ------------------------------------------
X_train_text = get_final_text(X_train)
X_kaggle_text = get_final_text(X_kaggle)

print("✓ Text fields extracted for MiniLM")

# ------------------------------------------
# Generate embeddings
# ------------------------------------------
print("🔄 Generating MiniLM embeddings for train...")
X_train_vectors = get_miniLM_embeddings(X_train_text)

print("🔄 Generating MiniLM embeddings for Kaggle test...")
X_kaggle_vectors = get_miniLM_embeddings(X_kaggle_text)

# ------------------------------------------
# Merge embeddings with other features
# ------------------------------------------
EMBEDDING_DIM = X_train_vectors.shape[1]
embed_cols = [f'miniLM_e_{i}' for i in range(EMBEDDING_DIM)]

X_train_nlp = pd.DataFrame(X_train_vectors, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vectors, columns=embed_cols)

X_train_advanced = pd.concat(
    [X_train_advanced.reset_index(drop=True), X_train_nlp.reset_index(drop=True)],
    axis=1
)
X_kaggle_advanced = pd.concat(
    [X_kaggle_advanced.reset_index(drop=True), X_kaggle_nlp.reset_index(drop=True)],
    axis=1
)

print("\n" + "=" * 50)
print("✓ MiniLM embeddings fused with features")
print("=" * 50)
print(f"Total features in training set: {X_train_advanced.shape[1]}")
print(f"Total features in Kaggle set: {X_kaggle_advanced.shape[1]}")

# ------------------------------------------
# XGBoost Hyperparameters
# ------------------------------------------
params = {
    'n_estimators': [300],
    'learning_rate': [0.1],
    'max_depth': [6, 10, 15],
    'subsample': [0.9],
    'colsample_bytree': [0.8, 0.9],
    'gamma': [0.5]
}

xgb_model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    tree_method='hist'
)

random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=params,
    n_iter=3,
    scoring='accuracy',
    cv=4,
    verbose=1,
    n_jobs=4,
    random_state=42
)

print("\n🔄 Starting Randomized Search with XGBoost...")
random_search.fit(X_train_advanced, y_train_clean)

print(f"\n🏆 Best CV Accuracy: {random_search.best_score_:.4f}")

# ------------------------------------------
# Predict Kaggle test
# ------------------------------------------
best_model = random_search.best_estimator_
y_pred_kaggle = best_model.predict(X_kaggle_advanced)

# ------------------------------------------
# Export submission
# ------------------------------------------
output = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": y_pred_kaggle
})

output.to_csv('submission_xgboost_advanced_final.csv', index=False)

print("\n✓ Submission file generated: submission_xgboost_advanced_final.csv")
print(output.head())


✓ Text fields extracted for MiniLM
🔄 Generating MiniLM embeddings for train...
🔄 Generating MiniLM embeddings for Kaggle test...

✓ MiniLM embeddings fused with features
Total features in training set: 417
Total features in Kaggle set: 417

🔄 Starting Randomized Search with XGBoost...
Fitting 4 folds for each of 3 candidates, totalling 12 fits


In [ ]:
from google.colab import files
files.download('submission_xgboost_advanced_final.csv')

In [12]:
# ==========================================
# LIGHTWEIGHT MiniLM + XGBoost PIPELINE (still unfinetuned)
# Memory optimized for Google Colab
# ==========================================

from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import numpy as np
import xgboost as xgb

# ------------------------------------------
# LOAD LIGHTER MiniLM (256-dim)
# ------------------------------------------

model_name = "sentence-transformers/all-MiniLM-L6-v2"
# 384-dim → still small & much lighter than 768-dim models

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
model.to("cuda")

# FP16 to cut memory usage by HALF
model.half()

# ------------------------------------------
# Efficient Mean Pooling
# ------------------------------------------
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).half()
    masked = token_embeddings * mask
    return masked.sum(1) / mask.sum(1).clamp(min=1e-9)

# ------------------------------------------
# Cleanup text (low cost)
# ------------------------------------------
def get_text(df):
    if "extended_tweet.full_text" in df.columns:
        return df["extended_tweet.full_text"].fillna("").astype(str)
    if "text" in df.columns:
        return df["text"].fillna("").astype(str)
    return pd.Series([""] * len(df))

# ------------------------------------------
# Lightweight Embedding Generator
# ------------------------------------------
def embed_text(text_series, batch_size=32):
    embeddings = np.zeros((len(text_series), 384), dtype=np.float16)

    for i in range(0, len(text_series), batch_size):
        batch = list(text_series.iloc[i:i+batch_size])
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=64,          # cut memory in HALF
        ).to("cuda")

        with torch.no_grad():
            out = model(**inputs)

        pooled = mean_pooling(out, inputs["attention_mask"])

        embeddings[i:i+batch_size] = pooled.float().cpu().numpy()

        torch.cuda.empty_cache()  # free VRAM

    return embeddings

# ------------------------------------------
# Get text data
# ------------------------------------------
X_train_text = get_text(X_train)
X_kaggle_text = get_text(X_kaggle)

print("✓ Text extracted")

# ------------------------------------------
# Generate embeddings
# ------------------------------------------
print("🔄 Embedding train...")
X_train_vec = embed_text(X_train_text)

print("🔄 Embedding Kaggle...")
X_kaggle_vec = embed_text(X_kaggle_text)

# ------------------------------------------
# Build DataFrames
# ------------------------------------------
embed_cols = [f"e{i}" for i in range(X_train_vec.shape[1])]

X_train_nlp = pd.DataFrame(X_train_vec, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vec, columns=embed_cols)

X_train_final = pd.concat(
    [X_train_advanced.reset_index(drop=True), X_train_nlp], axis=1
)
X_kaggle_final = pd.concat(
    [X_kaggle_advanced.reset_index(drop=True), X_kaggle_nlp], axis=1
)

print("✓ Embeddings merged")

# ------------------------------------------
# Lightweight XGBoost (fast & low memory)
# ------------------------------------------
xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

print("🔄 Training XGBoost...")
xgb_model.fit(X_train_final, y_train_clean)

# ------------------------------------------
# Predict Kaggle test
# ------------------------------------------
preds = xgb_model.predict(X_kaggle_final)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": preds
})

submission.to_csv("submission_lightweight.csv", index=False)

print("\n📄 Saved submission_lightweight.csv")
submission.head()

from google.colab import files
files.download('submission_lightweight.csv')


✓ Text extracted
🔄 Embedding train...
🔄 Embedding Kaggle...
✓ Embeddings merged
🔄 Training XGBoost...

📄 Saved submission_lightweight.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
#using lora finetuned weights
BASE_MODEL = "nreimers/MiniLM-L6-H384-uncased"
ADAPTER_PATH = "/content/content/minilm_lora_adapter"

# ---------------------------------------
# Load tokenizer & MODEL-FOR-SEQ-CLS
# ---------------------------------------
print("Loading tokenizer and classification base model...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    torch_dtype=torch.float16
)

# ---------------------------------------
# Load adapter (same architecture!)
# ---------------------------------------
print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.to("cuda")
model.eval()

print("✓ Model loaded on", next(model.parameters()).device)

# ---------------------------------------
# Mean Pooling
# ---------------------------------------
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.hidden_states[-1]   # <-- BETTER: use hidden states
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).to(token_embeddings.dtype)
    return (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

# Ask model to return hidden states
model.config.output_hidden_states = True

# ---------------------------------------
# Embedding function
# ---------------------------------------
def embed_text(text_series, batch_size=32):
    embeddings = np.zeros((len(text_series), 384), dtype=np.float16)

    for i in range(0, len(text_series), batch_size):
        batch = text_series.iloc[i:i+batch_size].tolist()

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            out = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                output_hidden_states=True
            )

        pooled = mean_pooling(out, enc["attention_mask"])

        embeddings[i:i+batch_size] = pooled.float().cpu().numpy()
        torch.cuda.empty_cache()

    return embeddings

# ------------------------------------------
# Get text data
# ------------------------------------------
X_train_text = get_text(X_train)
X_kaggle_text = get_text(X_kaggle)

print("✓ Text extracted")

# ------------------------------------------
# Generate embeddings
# ------------------------------------------
print("🔄 Embedding train...")
X_train_vec = embed_text(X_train_text)

print("🔄 Embedding Kaggle...")
X_kaggle_vec = embed_text(X_kaggle_text)

# ------------------------------------------
# Build DataFrames
# ------------------------------------------
embed_cols = [f"e{i}" for i in range(X_train_vec.shape[1])]

X_train_nlp = pd.DataFrame(X_train_vec, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vec, columns=embed_cols)

X_train_final = pd.concat(
    [X_train_advanced.reset_index(drop=True), X_train_nlp], axis=1
)
X_kaggle_final = pd.concat(
    [X_kaggle_advanced.reset_index(drop=True), X_kaggle_nlp], axis=1
)

print("✓ Embeddings merged")

# ------------------------------------------
# Lightweight XGBoost (fast & low memory)
# ------------------------------------------
xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

print("🔄 Training XGBoost...")
xgb_model.fit(X_train_final, y_train_clean)

# ------------------------------------------
# Predict Kaggle test
# ------------------------------------------
preds = xgb_model.predict(X_kaggle_final)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": preds
})

submission.to_csv("submission_lightweight.csv", index=False)

print("\n📄 Saved submission_lightweight.csv")
submission.head()

from google.colab import files
files.download('submission_lightweight.csv')


Loading tokenizer and classification base model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nreimers/MiniLM-L6-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading LoRA adapter...
✓ Model loaded on cuda:0
✓ Text extracted
🔄 Embedding train...
🔄 Embedding Kaggle...
✓ Embeddings merged
🔄 Training XGBoost...

📄 Saved submission_lightweight.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:

# ==========================================================
# 1) Load the stronger embedding model (BGE-small-en-v1.5)
# ==========================================================
import torch
import numpy as np
import pandas as pd
from pandas import json_normalize
from sentence_transformers import SentenceTransformer
import xgboost as xgb

MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("🔍 Loading BGE-small-en-v1.5...")
model = SentenceTransformer(MODEL_NAME, device="cuda")
embed_dim = model.get_sentence_embedding_dimension()
print("✓ Model loaded on GPU — embedding dim:", embed_dim)


# ==========================================================
# 2) Extract text safely
# ==========================================================
def get_text(df):
    if "extended_tweet.full_text" in df.columns:
        return df["extended_tweet.full_text"].fillna("").astype(str)
    if "text" in df.columns:
        return df["text"].fillna("").astype(str)
    return pd.Series([""] * len(df))


# ==========================================================
# 3) Embedding function (fast, Colab-friendly)
# ==========================================================
def embed_text(text_series, batch_size=64):
    texts = text_series.tolist()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cuda"    # ensures GPU inference
    )

    return embeddings.astype(np.float16)


# ==========================================================
# 4) Apply to your Kaggle & training data
# ==========================================================
print("🔄 Extracting text...")
X_train_text = get_text(X_train)
X_kaggle_text = get_text(X_kaggle)

print("🔄 Embedding train...")
X_train_vec = embed_text(X_train_text)

print("🔄 Embedding Kaggle...")
X_kaggle_vec = embed_text(X_kaggle_text)


# ==========================================================
# 5) Combine embeddings with your numerical features
# ==========================================================
embed_cols = [f"e{i}" for i in range(X_train_vec.shape[1])]

X_train_nlp = pd.DataFrame(X_train_vec, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vec, columns=embed_cols)

X_train_final = pd.concat([X_train_advanced.reset_index(drop=True),
                           X_train_nlp], axis=1)

X_kaggle_final = pd.concat([X_kaggle_advanced.reset_index(drop=True),
                            X_kaggle_nlp], axis=1)

print("✓ Embeddings merged with advanced features")


# ==========================================================
# 6) Lightweight XGBoost for final prediction
# ==========================================================
xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

print("🔄 Training XGBoost...")
xgb_model.fit(X_train_final, y_train_clean)

# ==========================================================
# 7) Predict & export submission
# ==========================================================
preds = xgb_model.predict(X_kaggle_final)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": preds
})

submission.to_csv("submission_bge_small.csv", index=False)

print("\n📄 Saved submission_bge_small.csv")
submission.head()

from google.colab import files
files.download('submission_bge_small.csv')

🔍 Loading BGE-small-en-v1.5...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Model loaded on GPU — embedding dim: 384
🔄 Extracting text...
🔄 Embedding train...
🔄 Embedding Kaggle...
✓ Embeddings merged with advanced features
🔄 Training XGBoost...

📄 Saved submission_bge_small.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# ==========================================================
# 1) Load BERTWEET for Embeddings
# ==========================================================
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd

MODEL_NAME = "vinai/bertweet-base"

print("🔍 Loading BERTweet...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModel.from_pretrained(MODEL_NAME).to("cuda")
model.eval()

print("✓ BERTweet ready")


# ==========================================================
# 2) Extract text safely
# ==========================================================
def get_text(df):
    if "extended_tweet.full_text" in df.columns:
        return df["extended_tweet.full_text"].fillna("").astype(str)
    if "text" in df.columns:
        return df["text"].fillna("").astype(str)
    return pd.Series([""] * len(df))


# ==========================================================
# 3) Embedding function (correct version – NO .encode())
# ==========================================================
def embed_text(text_series, batch_size=32):
    texts = text_series.tolist()
    all_emb = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            outputs = model(**inputs)
            # Mean pooling
            emb = outputs.last_hidden_state.mean(dim=1)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)

        all_emb.append(emb.cpu().numpy())

        torch.cuda.empty_cache()

    return np.concatenate(all_emb, axis=0).astype(np.float16)


# ==========================================================
# 4) Build Your Other Features (Advanced + NLP)
# ==========================================================
print("🛠️ Construction des features avancées (Métadonnées)...")

import nltk
nltk.download('stopwords')

# Provided by your earlier code
X_train_advanced = create_advanced_features(X_train)
X_kaggle_advanced = create_advanced_features(X_kaggle)

y_train_clean = y_train.astype(int)

# Provided by your earlier code
X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)

# Merge advanced + NLP features
X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

print(f"\nFeatures combinées ({len(X_train_advanced.columns)}):")
print(list(X_train_advanced.columns))


# ==========================================================
# 5) Embed Raw Text with BERTWEET
# ==========================================================
print("🔄 Extracting text...")
X_train_text = get_text(X_train)
X_kaggle_text = get_text(X_kaggle)

print("🔄 Embedding train...")
X_train_vec = embed_text(X_train_text)

print("🔄 Embedding Kaggle...")
X_kaggle_vec = embed_text(X_kaggle_text)

# Convert to DataFrame
embed_cols = [f"e{i}" for i in range(X_train_vec.shape[1])]
X_train_embed = pd.DataFrame(X_train_vec, columns=embed_cols)
X_kaggle_embed = pd.DataFrame(X_kaggle_vec, columns=embed_cols)

# Merge all features
X_train_final = pd.concat([X_train_advanced.reset_index(drop=True),
                           X_train_embed], axis=1)

X_kaggle_final = pd.concat([X_kaggle_advanced.reset_index(drop=True),
                            X_kaggle_embed], axis=1)

print("✓ All embeddings merged with advanced + NLP features")


# ==========================================================
# 6) Train Lightweight XGBoost
# ==========================================================
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

print("🔄 Training XGBoost...")
xgb_model.fit(X_train_final, y_train_clean)


# ==========================================================
# 7) Predict & Export Submission
# ==========================================================
preds = xgb_model.predict(X_kaggle_final)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": preds
})

submission.to_csv("submission_tweetransformer_small.csv", index=False)

print("\n📄 Saved submission_tweetransformer_small.csv")
submission.head()

from google.colab import files
files.download('submission_tweetransformer_small.csv')



🔍 Loading BERTweet...


vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


...

  -> Calcul du TF-IDF sur les Bios et entraînement du Méta-Modèle...


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


  -> Extraction du Sentiment (Polarity/Subjectivity) des Tweets...

Features combinées (33):
['user.followers_count', 'user.friends_count', 'user.listed_count', 'user.favourites_count', 'user.statuses_count', 'retweet_count', 'favorite_count', 'quote_count', 'reply_count', 'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days', 'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement', 'is_verified', 'is_default_profile', 'is_default_image', 'is_protected', 'has_url', 'tweet_hour', 'tweet_is_weekend', 'text_length', 'bio_length', 'num_urls', 'num_hashtags', 'num_mentions', 'has_media', 'meta_bio_proba', 'tweet_polarity', 'tweet_subjectivity', 'meta_tweet_proba']
🔄 Extracting text...
🔄 Embedding train...
🔄 Embedding Kaggle...
✓ All embeddings merged with advanced + NLP features
🔄 Training XGBoost...

📄 Saved submission_tweetransformer_small.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>